In [1]:
# ============================================================
# Notebook : 02_pcmci.ipynb
# Purpose  : Step 3 (Causal Stability) - part 1: temporal causal discovery.
#            (A) Run PCMCIplus (ParCorr, single lag) on the ten Track-A
#                sensor channels per household to recover the sensor-level
#                causal graph; the cross-variable links are weak by design.
#            (B) Identify, for each abnormal category, its lag-1 sensor
#                precursor using the same partial-correlation (ParCorr)
#                conditional-independence statistic PCMCI employs, with the
#                conditioning set = the other Track-A sensors at t-1 AND the
#                category's own autoregressive term at t-1. This recovers the
#                signal that PRECEDES each category (not the channel that
#                defines it contemporaneously): SkinTemperature precedes the
#                Skin-temp category, and the precursor set constrains Step 4.
# Note     : only sessions >= 24 h (144 steps) feed discovery; partial
#            correlations are weak (|r| <= ~0.13), so only their sign and
#            rank are meant to reproduce, not their magnitude.
# Design   : Single code cell. No absolute path is printed.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "axes.unicode_minus": False,
    "savefig.dpi": 600, "figure.dpi": 120, "axes.edgecolor": "0.15", "grid.color": "0.85",
})


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
S = meta["sensor_cont"]                 # 10 Track-A sensors
ABN = meta["abnormal_categories"]
CAT_LABEL = {"HR": "Heart rate", "BR": "Breath rate", "ST": "Skin temp.", "SP": "SpO2", "ER": "Emergency"}
TAU_MAX, PC_ALPHA, RECUR = 1, 0.05, 0.25

elig = df[df["pcmci_eligible"] == 1]
used_hh = [hh for hh, g in elig.groupby("HH_ID") if len(g) >= 144]
n_used = len(used_hh)

# ============================================================
# (A) PCMCIplus sensor causal graph (lag-1 cross-variable links).
# ============================================================
sensor_links = {}     # (src,tgt)->[parcorr]  (src!=tgt)
for hh in used_hh:
    g = elig[elig["HH_ID"] == hh].sort_values("Timestamp")
    X = g[S].astype(float).values
    keep = X.std(axis=0) > 1e-8
    names = [v for v, k in zip(S, keep) if k]
    Xs = (X[:, keep] - X[:, keep].mean(0)) / X[:, keep].std(0)
    res = PCMCI(dataframe=pp.DataFrame(Xs, var_names=names), cond_ind_test=ParCorr(),
                verbosity=0).run_pcmciplus(tau_min=0, tau_max=TAU_MAX, pc_alpha=PC_ALPHA)
    graph, vm = res["graph"], res["val_matrix"]
    for i in range(len(names)):
        for j in range(len(names)):
            if i != j and graph[i, j, 1] == "-->":
                sensor_links.setdefault((names[i], names[j]), []).append(vm[i, j, 1])

sl_rows = [{"source": k[0], "target": k[1], "freq_hh": len(v),
            "freq_share": round(len(v) / n_used, 3),
            "mean_parcorr": round(float(np.mean(v)), 4),
            "mean_abs": round(float(np.mean(np.abs(v))), 4)} for k, v in sensor_links.items()]
sensor_tbl = pd.DataFrame(sl_rows).sort_values(["freq_hh", "mean_abs"], ascending=False).reset_index(drop=True)
sensor_tbl.to_csv(TAB / "table_sensor_cross_links.csv", index=False, encoding="utf-8-sig")
max_sensor_cross = sensor_tbl["mean_abs"].max() if len(sensor_tbl) else float("nan")
recurring_sensor = sensor_tbl[sensor_tbl["freq_share"] >= RECUR]

# ============================================================
# (B) Category lag-1 precursors (ParCorr, conditioning on other sensors
#     at t-1 and the category's own term at t-1).
# ============================================================
def partial_corrs(y, X):
    """Partial correlation of each column of X with y (via OLS t-statistics)."""
    n = len(y)
    Xd = np.column_stack([np.ones(n), X])
    beta, _, _, _ = np.linalg.lstsq(Xd, y, rcond=None)
    resid = y - Xd @ beta
    dof = n - Xd.shape[1]
    s2 = (resid @ resid) / dof
    inv = np.linalg.pinv(Xd.T @ Xd)
    se = np.sqrt(np.maximum(np.diag(inv) * s2, 1e-18))
    tv = beta / se
    return (tv / np.sqrt(tv ** 2 + dof))[1:]


cat_prec = {c: {} for c in ABN}
for hh in used_hh:
    g = elig[elig["HH_ID"] == hh].sort_values("Timestamp")
    X = g[S].astype(float).values
    keep = X.std(axis=0) > 1e-8
    names = [v for v, k in zip(S, keep) if k]
    Xl = X[:-1][:, keep]
    Xl = (Xl - Xl.mean(0)) / Xl.std(0)
    for c in ABN:
        yv = g["cat_" + c].astype(float).values
        y, ylag = yv[1:], yv[:-1]
        if y.std() < 1e-8 or ylag.std() < 1e-8:
            continue
        Xc = np.column_stack([Xl, (ylag - ylag.mean()) / ylag.std()])
        pc = partial_corrs(y, Xc)[:-1]   # drop the autoregressive term's own coef
        for nm, v in zip(names, pc):
            cat_prec[c].setdefault(nm, []).append(v)

rows, prec_rows, causal_paths = [], [], {}
for c in ABN:
    ranked = sorted(cat_prec[c].items(), key=lambda kv: -abs(np.mean(kv[1])))
    for s, vals in ranked:
        rows.append({"category": c, "precursor_sensor": s, "freq_hh": len(vals),
                     "freq_share": round(len(vals) / n_used, 3),
                     "mean_parcorr": round(float(np.mean(vals)), 4),
                     "direction": "low->risk" if np.mean(vals) < 0 else "high->risk"})
    # causal-path = precursors that are both recurrent (present in a majority
    # of households) AND non-negligible in strength (|mean partial corr| >= 0.05)
    strong = [s for s, vals in ranked
              if len(vals) / n_used >= 0.5 and abs(np.mean(vals)) >= 0.05]
    path = strong if strong else ([ranked[0][0]] if ranked else [])
    causal_paths[c] = path
    if ranked:
        ts, tv = ranked[0][0], float(np.mean(ranked[0][1]))
        prec_rows.append({"category": c, "top_precursor": ts, "lag": "1 step (10 min)",
                          "mean_parcorr": round(tv, 4),
                          "direction": "low->risk" if tv < 0 else "high->risk",
                          "causal_path": ", ".join(path)})
    else:
        prec_rows.append({"category": c, "top_precursor": "(none)", "lag": "-",
                          "mean_parcorr": np.nan, "direction": "-", "causal_path": "(none)"})

links_tbl = pd.DataFrame(rows)
links_tbl.to_csv(TAB / "table_all_category_links.csv", index=False, encoding="utf-8-sig")
precursors_tbl = pd.DataFrame(prec_rows)
precursors_tbl.to_csv(TAB / "table_category_precursors.csv", index=False, encoding="utf-8-sig")
max_cat_abs = max((abs(np.mean(v)) for c in ABN for v in cat_prec[c].values()), default=float("nan"))
pd.to_pickle({"causal_paths": causal_paths, "used_hh": used_hh,
              "max_cat_abs": max_cat_abs, "max_sensor_cross": max_sensor_cross},
             DATA / "causal_paths.pkl")

# ============================================================
# Figure 5. Category <- sensor lag-1 precursor strength (signed annot), grayscale.
# ============================================================
M = pd.DataFrame(0.0, index=[CAT_LABEL[c] for c in ABN], columns=S)
for c in ABN:
    for s, vals in cat_prec[c].items():
        M.loc[CAT_LABEL[c], s] = np.mean(vals)
fig, ax = plt.subplots(figsize=(8.0, 3.0))
sns.heatmap(M.abs(), cmap="Greys", vmin=0, vmax=max(0.13, M.abs().values.max()),
            linewidths=0.4, linecolor="0.7", cbar_kws={"label": "|lag-1 partial corr|"},
            annot=M.round(3), fmt="", annot_kws={"size": 6}, ax=ax)
ax.set_xlabel("Precursor sensor (t-1)")
ax.set_ylabel("Abnormal category (t)")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=8)
save_fig(fig, "fig05_category_precursors")

# ============================================================
# Figure 6. Top precursor -> category graph (grayscale, bipartite).
# ============================================================
G = nx.DiGraph()
for r in prec_rows:
    if r["top_precursor"] != "(none)":
        G.add_edge(r["top_precursor"], CAT_LABEL[r["category"]],
                   w=abs(r["mean_parcorr"]) if r["mean_parcorr"] == r["mean_parcorr"] else 0.02)
fig, ax = plt.subplots(figsize=(7.0, 4.4))
sens_nodes = sorted(n for n in G.nodes if n in S)
cat_nodes = [n for n in G.nodes if n not in S]
pos = {n: (0, -i) for i, n in enumerate(sens_nodes)}
for i, n in enumerate(cat_nodes):
    pos[n] = (1.8, -i * (max(1, len(sens_nodes)) / max(1, len(cat_nodes))) - 0.5)
nx.draw_networkx_nodes(G, pos, nodelist=sens_nodes, node_color="0.9", edgecolors="black",
                       node_shape="o", node_size=1200, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=cat_nodes, node_color="0.45", edgecolors="black",
                       node_shape="s", node_size=1700, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
ws = [G[u][v]["w"] for u, v in G.edges()]
nx.draw_networkx_edges(G, pos, width=[0.6 + 16 * w for w in ws],
                       edge_color=[str(max(0.0, 0.7 - 3 * w)) for w in ws], arrowsize=13, ax=ax)
ax.axis("off")
save_fig(fig, "fig06_precursor_graph")

# ---- Console summary ----
print("=== Step 3 (part 1): causal discovery ===")
print(f"Households used (sessions >= 144 steps): {n_used}")
print(f"(A) PCMCIplus sensor graph: max cross-link |partial corr| = {max_sensor_cross:.4f} (weak)")
print(f"    recurring sensor cross-links (>= 25% hh): {len(recurring_sensor)}")
print(f"(B) Category precursors: max |partial corr| = {max_cat_abs:.4f}  (cf. weak, sign/rank focus)")
print()
print("Per-category top lag-1 precursor (Table 6 analogue):")
print(precursors_tbl.to_string(index=False))
print()
print("Figures saved: fig05_category_precursors, fig06_precursor_graph  (png + pdf, 600 dpi)")
print("Tables saved : table_category_precursors, table_all_category_links, table_sensor_cross_links")
print("Artifact     : causal_paths.pkl")


=== Step 3 (part 1): causal discovery ===
Households used (sessions >= 144 steps): 22
(A) PCMCIplus sensor graph: max cross-link |partial corr| = 0.1477 (weak)
    recurring sensor cross-links (>= 25% hh): 9
(B) Category precursors: max |partial corr| = 0.1303  (cf. weak, sign/rank focus)

Per-category top lag-1 precursor (Table 6 analogue):
category   top_precursor             lag  mean_parcorr  direction                  causal_path
      HR     Illuminance 1 step (10 min)       -0.1125  low->risk                  Illuminance
      BR     Illuminance 1 step (10 min)       -0.0890  low->risk                  Illuminance
      ST SkinTemperature 1 step (10 min)       -0.1303  low->risk SkinTemperature, Illuminance
      SP            SPO2 1 step (10 min)       -0.0523  low->risk                         SPO2
      ER     Activity_IR 1 step (10 min)        0.0915 high->risk                  Activity_IR

Figures saved: fig05_category_precursors, fig06_precursor_graph  (png + pdf, 600 dpi)